# Enterprise Fine-Tuning on Amazon Bedrock
### Production-Grade Custom Model Training Pipeline

This notebook covers an end-to-end, production-ready fine-tuning workflow on Amazon Bedrock using Cohere `command-light-text-v14` for clinical note generation.

**Key differences from foundations template:**
- Environment-based config (no hardcoded values)
- Centralized logging
- Full error handling and retry logic
- Automated IAM role management with least-privilege policy
- Job status polling with timeout
- Inference with streaming support
- Cleanup automation

## Prerequisites
- Amazon Bedrock access with Cohere `command-light-text-v14` enabled
- IAM permissions: `bedrock:*`, `s3:*`, `iam:CreateRole`, `iam:PutRolePolicy`
- Python 3.10+ with: `boto3`, `pandas`, `python-dotenv`
- `.env` file (or environment variables) with AWS config

## Step 1: Setup & Configuration

In [ ]:
!pip install -q boto3 pandas python-dotenv

import os
import json
import time
import logging
from dataclasses import dataclass
from typing import Optional

import pandas as pd
import boto3
from botocore.exceptions import ClientError
from dotenv import load_dotenv

load_dotenv()

logging.basicConfig(level=logging.INFO, format='%(asctime)s | %(levelname)s | %(message)s')
log = logging.getLogger('bedrock-finetune')

@dataclass
class Config:
    region:        str = os.getenv('AWS_REGION', 'us-east-1')
    bucket:        str = os.getenv('FINETUNE_S3_BUCKET', 'bedrock-finetuning-enterprise')
    dataset_path:  str = os.getenv('DATASET_PATH', 'dataset/data.csv')
    base_model_id: str = os.getenv('BASE_MODEL_ID', 'cohere.command-light-text-v14:7:4k')
    job_name:      str = os.getenv('FINETUNE_JOB_NAME', f'enterprise-finetune-{int(time.time())}')
    model_name:    str = os.getenv('CUSTOM_MODEL_NAME', f'enterprise-model-{int(time.time())}')
    role_name:     str = os.getenv('IAM_ROLE_NAME', 'BedrockEnterpriseFinetuneRole')
    epoch_count:   str = os.getenv('EPOCH_COUNT', '5')
    batch_size:    str = os.getenv('BATCH_SIZE', '8')
    learning_rate: str = os.getenv('LEARNING_RATE', '0.00003')
    poll_interval: int = int(os.getenv('POLL_INTERVAL_SEC', '60'))
    poll_timeout:  int = int(os.getenv('POLL_TIMEOUT_SEC', '7200'))

cfg = Config()
log.info(f"Config loaded — region={cfg.region}, bucket={cfg.bucket}, job={cfg.job_name}")

## Step 2: Initialize AWS Clients

In [ ]:
session    = boto3.Session(region_name=cfg.region)
s3         = session.client('s3')
iam        = session.client('iam')
bedrock    = session.client('bedrock')
br_runtime = session.client('bedrock-runtime')

# Verify Bedrock access
try:
    bedrock.list_foundation_models(byCustomizationType='FINE_TUNING')
    log.info("✅ Bedrock client initialized successfully")
except ClientError as e:
    log.error(f"❌ Bedrock access error: {e}")
    raise

## Step 3: Data Preparation

In [ ]:
df = pd.read_csv(cfg.dataset_path)
log.info(f"Loaded {len(df)} records | Columns: {df.columns.tolist()}")

required_cols = {'dialogue', 'note'}
if not required_cols.issubset(df.columns):
    raise ValueError(f"Dataset must contain: {required_cols}. Found: {set(df.columns)}")

# Quality filter
df = df.dropna(subset=['dialogue', 'note'])
df = df[df['dialogue'].str.len() > 50]
df = df[df['note'].str.len() > 20]
log.info(f"After quality filter: {len(df)} records remain")

# Train/val split (80/20)
train_df = df.sample(frac=0.8, random_state=42)
val_df   = df.drop(train_df.index)
log.info(f"Train: {len(train_df)}, Validation: {len(val_df)}")

In [ ]:
def save_jsonl(df: pd.DataFrame, path: str, prefix: str = '') -> str:
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, 'w') as f:
        for _, row in df.iterrows():
            json.dump({
                'prompt': f'{prefix}Summarize the following medical consultation.\n\n{row["dialogue"]}',
                'completion': row['note']
            }, f)
            f.write('\n')
    log.info(f"Saved {len(df)} records → {path}")
    return path

train_path = save_jsonl(train_df, 'dataset/train.jsonl')
val_path   = save_jsonl(val_df,   'dataset/val.jsonl')

## Step 4: S3 Setup & Upload

In [ ]:
def ensure_bucket(bucket: str, region: str) -> None:
    try:
        s3.head_bucket(Bucket=bucket)
        log.info(f"Bucket {bucket} exists.")
    except ClientError as e:
        if e.response['Error']['Code'] == '404':
            kwargs = {'Bucket': bucket}
            if region != 'us-east-1':
                kwargs['CreateBucketConfiguration'] = {'LocationConstraint': region}
            s3.create_bucket(**kwargs)
            # Enable versioning for enterprise
            s3.put_bucket_versioning(Bucket=bucket, VersioningConfiguration={'Status': 'Enabled'})
            log.info(f"Created bucket {bucket} with versioning enabled.")
        else:
            raise

ensure_bucket(cfg.bucket, cfg.region)

def upload_file(local_path: str, s3_key: str) -> str:
    s3.upload_file(local_path, cfg.bucket, s3_key)
    uri = f's3://{cfg.bucket}/{s3_key}'
    log.info(f"Uploaded {local_path} → {uri}")
    return uri

train_s3_uri = upload_file(train_path, 'data/train.jsonl')
val_s3_uri   = upload_file(val_path,   'data/val.jsonl')

## Step 5: IAM Role (Least-Privilege)

In [ ]:
trust_policy = {
    'Version': '2012-10-17',
    'Statement': [{'Effect': 'Allow', 'Principal': {'Service': 'bedrock.amazonaws.com'}, 'Action': 'sts:AssumeRole'}]
}

s3_policy = {
    'Version': '2012-10-17',
    'Statement': [{
        'Effect': 'Allow',
        'Action': ['s3:GetObject', 's3:PutObject', 's3:ListBucket', 's3:GetBucketLocation'],
        'Resource': [f'arn:aws:s3:::{cfg.bucket}', f'arn:aws:s3:::{cfg.bucket}/*']
    }]
}

try:
    res = iam.create_role(RoleName=cfg.role_name,
                          AssumeRolePolicyDocument=json.dumps(trust_policy),
                          Description='Least-privilege role for Bedrock enterprise fine-tuning')
    role_arn = res['Role']['Arn']
    log.info(f"Created role: {role_arn}")
except iam.exceptions.EntityAlreadyExistsException:
    role_arn = iam.get_role(RoleName=cfg.role_name)['Role']['Arn']
    log.info(f"Using existing role: {role_arn}")

iam.put_role_policy(RoleName=cfg.role_name, PolicyName='S3Access',
                    PolicyDocument=json.dumps(s3_policy))
time.sleep(10)  # IAM propagation delay
log.info("IAM role ready.")

## Step 6: Submit Fine-Tuning Job

In [ ]:
bedrock.create_model_customization_job(
    customizationType='FINE_TUNING',
    jobName=cfg.job_name,
    customModelName=cfg.model_name,
    roleArn=role_arn,
    baseModelIdentifier=cfg.base_model_id,
    hyperParameters={
        'epochCount':    cfg.epoch_count,
        'batchSize':     cfg.batch_size,
        'learningRate':  cfg.learning_rate,
    },
    trainingDataConfig={'s3Uri': train_s3_uri},
    validationDataConfig={'validators': [{'s3Uri': val_s3_uri}]},
    outputDataConfig={'s3Uri': f's3://{cfg.bucket}/output/'}
)
log.info(f"✅ Job submitted: {cfg.job_name}")

## Step 7: Poll Job Status with Timeout

In [ ]:
start = time.time()
terminal_states = {'Completed', 'Failed', 'Stopped'}

while True:
    elapsed = time.time() - start
    status  = bedrock.get_model_customization_job(jobIdentifier=cfg.job_name)['status']
    log.info(f"[{int(elapsed//60):02d}m {int(elapsed%60):02d}s] Status: {status}")

    if status in terminal_states:
        break
    if elapsed > cfg.poll_timeout:
        log.warning("⚠️ Polling timed out. Check job status in AWS Console.")
        break

    time.sleep(cfg.poll_interval)

if status == 'Completed':
    log.info("🎉 Fine-tuning complete!")
elif status == 'Failed':
    details = bedrock.get_model_customization_job(jobIdentifier=cfg.job_name)
    log.error(f"❌ Job failed: {details.get('failureMessage', 'No message')}")

## Step 8: Inference

> **Enterprise Note**: Purchase **Provisioned Throughput** in the Bedrock Console (Custom Models → Models → Purchase Provisioned Throughput). For production, choose a 1-month or 6-month commitment for significant cost savings.

In [ ]:
custom_model_arn = os.getenv('CUSTOM_MODEL_ARN', 'YOUR_PROVISIONED_MODEL_ARN')  # ← Set via .env or replace

test_cases = [
    {
        'title': 'Hypertension Follow-up',
        'dialogue': """
[doctor] Good morning. How have you been since your last visit?
[patient] Mostly okay, but I've had fatigue and dizziness when standing.
[doctor] Any racing heart? Shortness of breath?
[patient] Heart racing occasionally. No SOB. I missed some beta-blocker doses.
[doctor] That explains it. Let's check BP and EKG. Labs showed elevated cholesterol and borderline A1c — we'll start a statin.
[patient] Okay. Also more leg swelling after work.
[doctor] I'll adjust your diuretic. Follow up in 2 weeks.
"""
    }
]

for tc in test_cases:
    body = {
        'prompt': f'Summarize the following medical consultation.\n\n{tc["dialogue"]}',
        'temperature': 0.2,
        'p': 0.95,
        'max_tokens': 200
    }
    try:
        resp   = br_runtime.invoke_model(modelId=custom_model_arn, body=json.dumps(body))
        result = json.loads(resp['body'].read().decode('utf-8'))
        print(f"\n{'='*60}\n📋 {tc['title']}\n{'='*60}")
        print(result['generations'][0]['text'])
    except ClientError as e:
        log.error(f"Inference failed for '{tc['title']}': {e}")

## Step 9: Cleanup

> ⚠️ **Remove Provisioned Throughput from the AWS Console to stop billing.** Bedrock Console → Inference → Provisioned Throughput → Select → Delete.

In [ ]:
CLEANUP_ENABLED = False  # ← Set to True to auto-clean

if CLEANUP_ENABLED:
    # Delete training data from S3
    for key in ['data/train.jsonl', 'data/val.jsonl']:
        s3.delete_object(Bucket=cfg.bucket, Key=key)
        log.info(f"Deleted s3://{cfg.bucket}/{key}")

    # Detach and delete IAM role policy
    iam.delete_role_policy(RoleName=cfg.role_name, PolicyName='S3Access')
    iam.delete_role(RoleName=cfg.role_name)
    log.info(f"Deleted IAM role: {cfg.role_name}")
else:
    print("⚠️  Cleanup skipped. Set CLEANUP_ENABLED=True to auto-clean.")
    print("  → Remember to remove Provisioned Throughput from the Bedrock Console!")